In [ ]:
# Kaggle setup (does nothing outside Kaggle). Needs: GPU on, Internet on, competition data as input.
# The repo stays private until the deadline, so it is cloned with a GitHub token saved as a
# Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets; fine-grained token, read-only, this repo only).
import os, subprocess
REPO = "polluvl/hw2-video-detection"   # <- set once, e.g. "polluvl/hw2-video-detection"
if os.path.isdir("/kaggle/working") and not os.path.isdir("/kaggle/working/repo"):
    try:
        from kaggle_secrets import UserSecretsClient
        auth = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
    except Exception:
        auth = ""  # no secret -> works once the repo is public
    r = subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
                       capture_output=True)
    if r.returncode != 0:  # do not print the command: it contains the token
        raise RuntimeError("git clone failed: check REPO and the GITHUB_TOKEN secret")
    # remove the token from the saved clone (outputs of /kaggle/working can be shared)
    subprocess.run(["git", "-C", "/kaggle/working/repo", "remote", "set-url", "origin",
                    f"https://github.com/{REPO}.git"], check=True)
    subprocess.run(["pip", "install", "-q", "-r", "/kaggle/working/repo/requirements.txt"], check=True)
if os.path.isdir("/kaggle/working/repo/notebooks"):
    os.chdir("/kaggle/working/repo/notebooks")

# 04 · One-stage anchor-free: YOLOv8 (Ultralytics, COCO-pretrained)
YOLOv8 has no anchors: every cell of the P3/P4/P5 feature maps predicts the distances from its
centre to the four box sides (with Distribution Focal Loss) plus class scores. Which cells are
positives for a GT box is decided by the Task-Aligned Assigner (TOOD): among cells whose centre
lies inside the box, take the top-10 by `score^α · IoU^β`.

Protocol: the same train/val frames as 02/03, ignored regions masked in training images.
Differences from the torchvision protocol (stated in the report): Ultralytics' own augmentation
(mosaic, HSV, scale, flip) and optimiser defaults, epochs instead of a fixed iteration count.
Final numbers come from OUR evaluator on the original val images (ignore regions = iscrowd).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from ultralytics import YOLO

sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("src").resolve()))
from analysis import error_breakdown, recall_by_size, summarise_errors
from common import (SMOKE, evaluate, find_data_root, list_frames, load_preds,
                    log_result, save_preds, seed_everything, work_dir)
from yolo_utils import (export_yolo_split, predict_ultralytics, read_curves,
                        write_data_yaml)

DATA = find_data_root()
OUT = work_dir()
seed_everything(42)
TRAIN_STEP, VAL_STEP = (1, 1) if SMOKE else (10, 5)
EPOCHS = 1 if SMOKE else 30
DEVICE = 0 if torch.cuda.is_available() else "cpu"

train_frames = list_frames(DATA / "train", step=TRAIN_STEP)
val_frames = list_frames(DATA / "val", step=VAL_STEP)

## Export to Ultralytics format (once; reused by 05 RT-DETR)

In [ ]:
YOLO_DS = OUT / f"yolo_ds_step{TRAIN_STEP}"
n1 = export_yolo_split(train_frames, YOLO_DS / "train", mask_ignore=True)
# val copy is also masked: it is only used for Ultralytics' per-epoch curves, which cannot handle
# ignore regions otherwise. Reported metrics always come from `evaluate()` on original images.
n2 = export_yolo_split(val_frames, YOLO_DS / "val", mask_ignore=True)
DATA_YAML = write_data_yaml(YOLO_DS / "data.yaml", YOLO_DS / "train", YOLO_DS / "val")
print(f"exported, masked images: train {n1}, val {n2}")

## Experiments
Search range: input resolution {640, 1024} and an extra high-resolution P2 head (stride 4).
- `yolo_640`     – baseline, YOLOv8s at 640 px (default)
- `yolo_1024`    – input 1024 px: a 20 px object in a 1920-wide frame becomes 6.7 px at 640
  but 10.7 px at 1024
- `yolo_1024_p2` – + P2 head (stride 4) so tiny objects get more grid cells (more positives)
- `yolo_1024_box` – loss modification: box-loss gain 7.5 → 10 (localisation matters more for tiny boxes)

In [ ]:
COMMON_ARGS = dict(data=DATA_YAML, epochs=EPOCHS, batch=2 if SMOKE else 8, workers=2, seed=42,
                   deterministic=True, device=DEVICE, project=str(OUT / "ultralytics"), exist_ok=True,
                   patience=100, plots=not SMOKE, cos_lr=True, close_mosaic=5, max_det=500)

RUNS = {
    "yolo_640": dict(model="yolov8s.pt", imgsz=640),
    "yolo_1024": dict(model="yolov8s.pt", imgsz=1024),
    "yolo_1024_p2": dict(model="yolov8s-p2.yaml", weights="yolov8s.pt", imgsz=1024),
    "yolo_1024_box": dict(model="yolov8s.pt", imgsz=1024, box=10.0),
}
if SMOKE:  # no internet / no GPU: build from yaml, tiny images
    for r in RUNS.values():
        r["model"] = r["model"].replace(".pt", ".yaml")
        r.pop("weights", None)
        r["imgsz"] = 320


def train_and_eval(name, spec):
    spec = dict(spec)
    model = YOLO(spec.pop("model"))
    if "weights" in spec:  # P2 model: transfer all layers that match from the COCO checkpoint
        model.load(spec.pop("weights"))
    model.train(name=name, **COMMON_ARGS, **spec)
    run_dir = OUT / "ultralytics" / name
    best = YOLO(str(run_dir / "weights" / "best.pt"))
    preds, ms = predict_ultralytics(best, val_frames, imgsz=spec["imgsz"])
    save_preds(preds, OUT / f"preds_val_{name}.npz")
    metrics = evaluate(val_frames, preds)
    curves = read_curves(run_dir)
    log_result(name, metrics, {"family": "one-stage anchor-free", "ms_per_img": round(ms, 1),
                               "train_min": round(curves["minutes"].iloc[-1], 1) if "minutes" in curves else None,
                               "config": str(spec)})
    print(name, metrics, f"{ms:.1f} ms/img")
    return metrics


for name, spec in RUNS.items():
    train_and_eval(name, spec)
    torch.cuda.empty_cache()

## Curves (Ultralytics internal val) and results

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name in RUNS:
    c = read_curves(OUT / "ultralytics" / name)
    ax.plot(c.epoch, c.mAP, label=name)
ax.set_xlabel("epoch")
ax.set_ylabel("mAP50-95 (Ultralytics val)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures_yolo_curves.png", dpi=120)
plt.show()

results = pd.read_csv(OUT / "results.csv")
cols = ["experiment", "family", "mAP", "AP50", "AP_small", "AP_medium", "AP_large", "AR", "ms_per_img"]
results[cols]

## Comparison by object size with the tuned anchor-based detectors

In [ ]:
def best_of(prefix):
    r = results[results.experiment.str.startswith(prefix)]
    return r.sort_values("mAP").experiment.iloc[-1] if len(r) else None

compare = [n for n in [best_of("frcnn"), best_of("retina"), "yolo_640", best_of("yolo")] if n]
size_recall = {}
for name in dict.fromkeys(compare):
    p = load_preds(OUT / f"preds_val_{name}.npz")
    gts, fps = error_breakdown(val_frames, p, score_thr=0.25)
    size_recall[name] = recall_by_size(gts)
print("recall at score>=0.25 by GT size:")
pd.DataFrame(size_recall)

TODO (your analysis): how are locations assigned without anchors (centre-inside + top-k TAL),
why that matters for tiny/dense objects (one stride-8 cell can lie inside several neighbouring
boxes; a P2 head halves the stride), and what the size table shows.